# DataFrame and SQL Table: Same Data, Two Shapes

**Module 4: Python Data Analysis**

**Learning Objectives:**
- Explain how a pandas DataFrame corresponds to a SQL table's rows, columns, and types
- Translate a small SQL query into its pandas equivalent, and say where each one actually runs
- Describe what `uv` is and why this module manages its packages with it instead of bare `pip`

**Skills:** DataFrame, Series, dtype, in-memory, uv


> **You are not presenting this notebook to students.** Use it as your own answer key while you fill in a student notebook live, in front of the class.


## Scenario

You have joined a city agency's data team. Every service request the city takes lives in a PostgreSQL database, and you already know how to ask it for rows: `SELECT unique_key, created_date, complaint_type, borough FROM service_requests LIMIT 5;`. Today those same rows reach you as a small CSV export of that table, because connecting to the live database for real is next class. Your job today is to see the same NYC 311 rows in both shapes, a SQL table and a pandas DataFrame, and learn exactly how the two line up.


## Before You Start

A SQL table declares a type on every column. When those same rows land in a DataFrame, where does the type information come from?

*Hint: Think about what a `.csv` file actually stores on disk. Is a date in a CSV any different from a piece of text?*


## Documentation

- The Same Rows, Now **In Your Own Memory**: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

- Every Column Has **One Type**, Just Like SQL: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html

- A `WHERE` Clause Is Just a **True/False Mask**: https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing


## The Same Rows, Now **In Your Own Memory**

*Context: You have seen NYC 311 data before, and that is deliberate. Today the new thing is the comparison, not the dataset.*

**Analogy:** Same album, one on a shelf you have to walk over to, one already downloaded on your phone. Identical tracks either way.

**Backup analogies** (if the primary doesn't land):

- **Sports:** The same game watched live in the stadium, or recorded on your phone, is still the same plays either way.
- **Community:** A recipe in your grandmother's book versus copied onto a card in your pocket, still the same recipe either way.

**Instructor notes:**

> [ACTIVITY] Have everyone run this cell and confirm they see five rows before moving on. Online: ask for a thumbs-up reaction rather than reading names out.
> 
> [SCRIPT] Name the missing step out loud before running anything: somebody exported those rows out of the database into a `.csv` file, and that export is the hop where the type declarations got left behind. The table came along. The schema did not.
> 
> [SCRIPT] Say the path assumption out loud once, here, and never again: the shared data folder sits two levels above the notebook, which is what `../../data/` means. Anyone who typed `data/` alone will get a `FileNotFoundError`, and that is the only path problem today should produce.
> 
> [SCRIPT] Point at the output and say it plainly: these are the same five rows as the previous slide, in the same order, with the same column names. `.head()` shows all seven columns, so three that the table left out show up here too. What changed is not the data, only where it is sitting.
> 
> [SCRIPT] Name the analogy before the term, then attach the term: that thing now sitting in memory has a name, and the name is a DataFrame. If the album analogy does not land, try one of the backups in this cell's notebook entry.
> 
> [MISCONCEPTIONS] Somebody will spot a `NaN` in the very first row of the output and ask about it. Have the answer ready, because it is on today's topic, not a distraction: the file genuinely contains the characters `N/A` in that cell, and `pd.read_csv` treats `N/A` as one of its default "this means missing" strings, so it becomes `NaN`. Nobody asked it to. That is the same idea as the type story coming next, arriving a slide early: reading a file means a tool making decisions about what the characters mean. Say that, then move on; real missing-value handling belongs to the project, not to today.
> 
> [MISCONCEPTIONS] A common belief is that a DataFrame is "a spreadsheet in Python." Close enough to be useful, wrong in one important way: a spreadsheet lets any cell hold anything, and a DataFrame gives the whole column one type, exactly like SQL does. The next slide makes that visible.

*Input:*


In [ ]:
import pandas as pd

# The same 311 rows the SQL table just showed
df = pd.read_csv("../../data/nyc_311_sample.csv")

print(df.head())


## Every Column Has **One Type**, Just Like SQL

**Analogy:** A tag reading `$24.99` is just characters until a register reads it into a field that expects money.

**Backup analogies** (if the primary doesn't land):

- **Gaming:** Your level typed into a chat message is only text. The game treats it as a number inside the game itself.
- **Travel:** A flight time on a napkin is text. Your calendar only counts down to it once you enter it as a real time.

**Instructor notes:**

> [SCRIPT] Walk the output column by column. `unique_key` came in as `int64`, a real integer. Every other column came in as text. Then land the important one: in the database, `created_date` was a real timestamp. In this DataFrame it is text, and it stays text until `pd.to_datetime()` is called.
> 
> [SCRIPT] Say why out loud: the database stores the type alongside the data because the schema declares it. A CSV stores only characters, so pandas has to guess a type per column when it reads one, and text is the safe guess. This is not pandas being careless. It is the type information genuinely not being in the file.
> 
> [SCRIPT] Read the printed label out loud rather than predicting it, because it depends on which pandas is installed. On pandas 2.x a text column prints as `object`. On pandas 3.0 and later it prints as `str`. Both mean text. If two people in the room see different labels, that is the real answer, not a bug, and it is a good moment to say why the lesson insists on running `.dtypes` instead of assuming.
> 
> [MISCONCEPTIONS] People read `object` as "something went wrong." It did not. It is just how pandas 2.x labels a column of text. It is the right label for `borough`, and the wrong one for a date, which is the whole reason to look.
> 
> [SOURCES] Verified by running this exact code on this exact file with pandas 2.2.3: `unique_key` is `int64` and the remaining six columns are `object`. The version-dependence is pandas 3.0.0 release notes, PDEP-14, dedicated string dtype enabled by default, released 2026-01-21, pandas.pydata.org/docs/whatsnew/v3.0.0.html.

*Input:*


In [ ]:
print(df.dtypes)

# A date arrives as text until you actually parse it
dates = pd.to_datetime(df["created_date"])
print(dates.dtype)


### 🤔 Check for Understanding: Types Don't Travel

*Mode: Poll*

`df.dtypes` shows `created_date` as text, but in the database that same column was a real `TIMESTAMP`. What happened?

- The file stored it as text, and nothing re-declared it as a date
- pandas cannot store dates at all
- The database exported the wrong column

**Answer:** The file stored it as text, and nothing re-declared it as a date — A column's type lives in the database's own schema, not in the rows. A CSV holds only characters, so pandas guesses a type per column as it reads, and text is the safe guess. Calling `pd.to_datetime()` is what turns it back into a real date.

**Instructor notes:**

> [ACTIVITY] Poll: read the three options, count hands on each, then reveal. Online: post the three options in chat as 1/2/3 and have everyone type a number. 45 seconds.
> 
> [SCRIPT] Option two is worth pausing on before dismissing it. pandas has a genuinely good datetime type, which is exactly why the fix is one function call rather than a workaround.


## A `WHERE` Clause Is Just a **True/False Mask**

**Analogy:** A guest list at the door, one pass marking each person yes or no, then a second pass letting only the yeses in.

**Backup analogies** (if the primary doesn't land):

- **Sports:** A coach reads the roster and calls out starter or bench for each name, then only the starters walk onto the court.
- **Fashion:** Sorting a closet by tagging every item keep or donate first, then carrying only the keep pile back to the rack.

**Instructor notes:**

> [ACTIVITY] Fill in the blank. Give 30 seconds before advancing to the answer. In person: ask someone to say the missing column name out loud. Online: everyone types it in chat at the same time.
> 
> [SCRIPT] After the reveal, slow down on what the inner part actually produces. `df["borough"] == "BRONX"` is not the filter. It is fifteen True/False values, one per row. Handing that back to `df[...]` is what keeps the True ones. That is exactly the job `WHERE` does inside the database, just written out in the open.
> 
> [SCRIPT] Point at the two printed numbers: three Bronx rows, and a count per complaint type. That is a `WHERE` and a `GROUP BY` in two lines of pandas.
> 
> [SCRIPT] Heads up on a real difficulty gap: the blank on this slide is one word, a 30-second whole-room beat. The same cell in the student notebook asks for all three lines, including the count and the grouping. That is deliberate, but do not assume finishing the slide means everyone can finish the notebook cell unaided.
> 
> [MISCONCEPTIONS] A very common early mistake is writing `df["borough" == "BRONX"]`, with the comparison inside the brackets. That compares two strings, gets `False`, and asks the DataFrame for a column named `False`. Show that error on purpose if time allows, it is the fastest way to make the two-step nature of a mask stick.

*Input:*


In [ ]:
bronx = df[df["borough"] == "BRONX"]

print(len(bronx), "Bronx requests")
print(bronx["complaint_type"].value_counts())
